# Advanced Model Evaluation & Outbreak Prediction
## Disease Surveillance Early Warning System

This notebook focuses on advanced model evaluation, community-centric analysis, and outbreak detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import sys
import os

# Add models to path
sys.path.insert(0, '../models')

from data_processor import DataProcessor
from model_builder import DiseasePredictor, DiseaseSeverityClassifier
from alert_system import AlertSystem, WeatherRiskAnalyzer
from community_reporting import CommunityReportingSystem, SymptomType

print("All libraries and modules imported successfully!")

## 1. Initialize All Systems

In [ ]:
# Initialize data processor
base_path = '..'
weather_path = os.path.join(base_path, 'data', 'weather_data.csv')
disease_path = os.path.join(base_path, 'data', 'disease_data.csv')

processor = DataProcessor(weather_path, disease_path)
processor.preprocess_weather_data()
processor.preprocess_disease_data()
processor.merge_data()
processor.create_features_for_prediction()
processor.scale_features()

print("✓ Data Processor initialized")

# Get training data
X_train, X_test, y_train, y_test = processor.get_training_data()

# Train models
predictor = DiseasePredictor()
predictor.train_all_models(X_train, y_train, X_test, y_test)

print("✓ Predictor Models trained")

# Initialize alert system
alert_system = AlertSystem(predictor)
print("✓ Alert System initialized")

# Initialize reporting system
reporting_system = CommunityReportingSystem()
print("✓ Community Reporting System initialized")

## 2. Detailed Model Evaluation

In [ ]:
# Print comprehensive model evaluation
predictor.print_model_summary()

# Get best model
best_model_name, best_model = predictor.get_best_model()
print(f"\n🏆 Best Model: {best_model_name}")

In [ ]:
# Create detailed evaluation metrics comparison
evaluation_df = pd.DataFrame(predictor.model_performance).T

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Train R²',
    x=evaluation_df.index,
    y=evaluation_df['train_r2']
))

fig.add_trace(go.Bar(
    name='Test R²',
    x=evaluation_df.index,
    y=evaluation_df['test_r2']
))

fig.update_layout(
    title='Model Performance Comparison - R² Scores',
    barmode='group',
    xaxis_title='Model',
    yaxis_title='R² Score'
)
fig.show()

## 3. Weather Risk Analysis

In [ ]:
# Simulate weather scenarios and analyze risk
weather_scenarios = {
    'High Risk (Monsoon)': {'temperature': 28, 'humidity': 85, 'rainfall': 120, 'is_monsoon': True, 'days_since_rain': 2},
    'Medium Risk (Summer)': {'temperature': 35, 'humidity': 55, 'rainfall': 10, 'is_monsoon': False, 'days_since_rain': 10},
    'Low Risk (Winter)': {'temperature': 18, 'humidity': 40, 'rainfall': 5, 'is_monsoon': False, 'days_since_rain': 15},
    'Vector Optimal (Tropical)': {'temperature': 26, 'humidity': 70, 'rainfall': 50, 'is_monsoon': False, 'days_since_rain': 4},
}

analyzer = WeatherRiskAnalyzer()
risk_results = {}

for scenario_name, weather_data in weather_scenarios.items():
    water_risk, water_factors = analyzer.assess_water_borne_risk(
        weather_data['temperature'],
        weather_data['humidity'],
        weather_data['rainfall'],
        weather_data['is_monsoon']
    )
    
    vector_risk, vector_factors = analyzer.assess_vector_borne_risk(
        weather_data['temperature'],
        weather_data['humidity'],
        weather_data['rainfall'],
        weather_data['days_since_rain']
    )
    
    risk_results[scenario_name] = {
        'Water-Borne': water_risk,
        'Vector-Borne': vector_risk
    }

# Visualize risk scenarios
risk_df = pd.DataFrame(risk_results).T

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Water-Borne Risk',
    x=risk_df.index,
    y=risk_df['Water-Borne']
))

fig.add_trace(go.Bar(
    name='Vector-Borne Risk',
    x=risk_df.index,
    y=risk_df['Vector-Borne']
))

fig.update_layout(
    title='Disease Risk by Weather Scenario',
    barmode='group',
    xaxis_title='Scenario',
    yaxis_title='Risk Score (0-100)',
    height=500
)
fig.show()

## 4. Alert Generation & Risk Assessment

In [ ]:
# Generate alerts for different locations
locations_weather = {
    'Mumbai': {'temperature': 32, 'humidity': 85, 'rainfall': 120, 'is_monsoon': True},
    'Chennai': {'temperature': 28, 'humidity': 75, 'rainfall': 95, 'is_monsoon': False},
    'Delhi': {'temperature': 38, 'humidity': 45, 'rainfall': 5, 'is_monsoon': False},
    'Kolkata': {'temperature': 26, 'humidity': 80, 'rainfall': 110, 'is_monsoon': True},
    'Bangalore': {'temperature': 24, 'humidity': 65, 'rainfall': 40, 'is_monsoon': False},
}

print("\n" + "="*80)
print("DISEASE SURVEILLANCE ALERTS")
print("="*80)

alerts = {}
for location, weather in locations_weather.items():
    alert = alert_system.generate_alert(
        location=location,
        weather_data=weather,
        predicted_cases=np.random.randint(50, 500),
        alert_type='both'
    )
    alerts[location] = alert
    
    print(f"\n📍 {location}")
    print(f"   Alert Level: {alert['overall_alert_level']['name']}")
    print(f"   Water-Borne Risk: {alert['risk_scores'].get('water_borne', 'N/A')}%")
    print(f"   Vector-Borne Risk: {alert['risk_scores'].get('vector_borne', 'N/A')}%")

In [ ]:
# Visualize alert distribution
alert_levels = [alert['overall_alert_level']['name'] for alert in alerts.values()]
alert_colors = {'Safe': 'green', 'Warning': 'yellow', 'Danger': 'orange', 'Critical': 'red'}

alert_summary = pd.Series(alert_levels).value_counts()

fig = px.pie(
    values=alert_summary.values,
    names=alert_summary.index,
    title='Alert Level Distribution Across Locations',
    color_discrete_map=alert_colors
)
fig.show()

## 5. Community Health Reports & Outbreak Detection

In [ ]:
# Simulate community health reports
print("\n" + "="*80)
print("SUBMITTING COMMUNITY HEALTH REPORTS")
print("="*80)

# Water-borne outbreak simulation
water_borne_reports = [
    ('reporter_001', 'Mumbai', ['Diarrhea', 'Vomiting', 'Fever with GI symptoms'], 1),
    ('reporter_002', 'Mumbai', ['Diarrhea', 'Abdominal Pain', 'Dehydration'], 2),
    ('reporter_003', 'Mumbai', ['Vomiting', 'Diarrhea', 'Abdominal Pain'], 1),
    ('reporter_004', 'Mumbai', ['Fever with GI symptoms', 'Diarrhea'], 3),
    ('reporter_005', 'Mumbai', ['Diarrhea', 'Abdominal Pain'], 2),
    ('reporter_006', 'Mumbai', ['Diarrhea', 'Dehydration'], 1),
]

# Vector-borne outbreak simulation
vector_borne_reports = [
    ('reporter_007', 'Chennai', ['High Fever', 'Joint Pain', 'Rash'], 1),
    ('reporter_008', 'Chennai', ['High Fever', 'Muscle Pain', 'Headache'], 1),
    ('reporter_009', 'Chennai', ['Joint Pain', 'Body Ache', 'Rash'], 2),
    ('reporter_010', 'Chennai', ['High Fever', 'Headache', 'Nausea'], 1),
    ('reporter_011', 'Chennai', ['Fever with GI symptoms', 'Joint Pain'], 1),
]

# Submit reports
for reporter_id, location, symptoms, affected in water_borne_reports + vector_borne_reports:
    report = reporting_system.submit_report(reporter_id, location, symptoms, affected)
    print(f"✓ Report {report.report_id[:8]}... submitted from {location}")

# Verify some reports
print(f"\nVerifying {len(reporting_system.reports)//2} reports...")
for i, report in enumerate(reporting_system.reports[:len(reporting_system.reports)//2]):
    reporting_system.verify_report(report.report_id, 'confirmed', 'health_worker_001')

In [ ]:
# Detect outbreaks
print("\n" + "="*80)
print("OUTBREAK DETECTION")
print("="*80)

outbreaks = reporting_system.detect_outbreaks(min_reports=3)

if outbreaks:
    for outbreak in outbreaks:
        print(f"\n🚨 OUTBREAK DETECTED")
        print(f"   Location: {outbreak['location']}")
        print(f"   Disease Type: {outbreak['disease_type']}")
        print(f"   Confirmed Cases: {outbreak['confirmed_cases']}")
        print(f"   Total Affected: {outbreak['total_affected']}")
        print(f"   Severity Score: {outbreak['severity']:.1f}/100")
        print(f"   Alert Level: {outbreak['alert_level']}")
else:
    print("No outbreaks detected")

In [ ]:
# Get summary statistics
summary = reporting_system.get_summary_report()

print("\n" + "="*80)
print("COMMUNITY REPORTING SUMMARY")
print("="*80)
print(f"Total Reports: {summary['total_reports']}")
print(f"Confirmed Cases: {summary['confirmed_cases']}")
print(f"Total Affected: {summary['total_affected']}")
print(f"Locations Affected: {summary['locations_with_reports']}")
print(f"Active Outbreak Signals: {summary['outbreak_signals']}")
print(f"Average Verification Score: {summary['avg_verification_score']:.2f}/100")

## 6. Location-wise Risk Analysis

In [ ]:
# Get statistics for each location
location_stats = []

for location in ['Mumbai', 'Chennai']:
    stats = reporting_system.get_location_statistics(location)
    if stats:
        location_stats.append({
            'Location': location,
            'Total Reports': stats['total_reports'],
            'Confirmed Cases': stats['confirmed_cases'],
            'Total Affected': stats['total_affected'],
            'Avg Verification': f"{stats['avg_verification_score']:.1f}"
        })

if location_stats:
    stats_df = pd.DataFrame(location_stats)
    print("\n" + "="*80)
    print("LOCATION-WISE STATISTICS")
    print("="*80)
    print(stats_df.to_string(index=False))

## 7. Predictions with Confidence Intervals

In [ ]:
# Make predictions with confidence intervals
sample_X = X_test.iloc[:5]
predictions, lower_ci, upper_ci = predictor.predict_with_confidence(sample_X, model_name='gradient_boosting')

print("\n" + "="*80)
print("PREDICTIONS WITH CONFIDENCE INTERVALS")
print("="*80)

for i, (pred, lower, upper) in enumerate(zip(predictions, lower_ci, upper_ci)):
    print(f"\nSample {i+1}:")
    print(f"  Predicted Cases: {pred:.0f}")
    print(f"  95% CI: [{lower:.0f}, {upper:.0f}]")
    
    # Determine risk level
    risk_level, confidence = DiseaseSeverityClassifier.classify_risk_level(pred)
    recommendations = DiseaseSeverityClassifier.get_recommendations(risk_level)
    
    print(f"  Risk Level: {risk_level}")
    print(f"  Confidence Score: {confidence:.1f}/100")

## 8. Comprehensive Risk Report

In [ ]:
# Generate comprehensive report
print("\n" + "="*80)
print("SMART HEALTH SURVEILLANCE PLATFORM")
print("COMPREHENSIVE DISEASE RISK ASSESSMENT REPORT")
print("="*80)
print(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# System status
print("\n📊 SYSTEM STATUS:")
print(f"  ✓ AI Models: {len(predictor.models)} trained and deployed")
print(f"  ✓ Locations Monitored: {len(locations_weather)}")
print(f"  ✓ Community Reports: {len(reporting_system.reports)}")
print(f"  ✓ Active Outbreaks: {len(outbreaks)}")

# Alert summary
print("\n⚠️  ALERT SUMMARY:")
alert_summary_final = alert_system.get_alert_summary()
for key, value in alert_summary_final.items():
    if key != 'timestamp':
        print(f"  {key.replace('_', ' ').title()}: {value}")

# Model performance
print("\n🤖 MODEL PERFORMANCE:")
best_metrics = predictor.model_performance[best_model_name]
print(f"  Best Model: {best_model_name}")
print(f"  Test R² Score: {best_metrics['test_r2']:.4f}")
print(f"  Test RMSE: {best_metrics['test_rmse']:.2f} cases")
print(f"  Test MAE: {best_metrics['test_mae']:.2f} cases")

# Key recommendations
print("\n✅ KEY RECOMMENDATIONS:")
print("  1. Intensify surveillance in high-risk locations")
print("  2. Increase water quality testing during monsoon")
print("  3. Deploy mobile health clinics in outbreak areas")
print("  4. Distribute health awareness materials")
print("  5. Strengthen community reporting mechanisms")

print("\n" + "="*80)